# Vectorstores and Embeddings

In [ ]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [ ]:
# 故意重复加载同一份 PDF 两次，模拟真实场景中常见的"脏数据"（重复文档），
# 后面可以看到相似度搜索时重复文档会导致检索结果里出现重复/冗余内容
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# chunk_size=1500 比第 02 节的示例大很多，因为这里是真实检索场景，块太小会丢失上下文
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150
)

In [ ]:
# TODO: 请在此处补全代码（用 text_splitter.split_documents(docs) 得到 splits）
splits = None  # TODO

In [ ]:
len(splits)

## Embeddings

In [ ]:
# 【真实 Bug 修复】OpenAIEmbeddings：把文本转换成高维向量（embedding），是 RAG 流程第三步的核心。
# 语义相近的句子，embedding 向量之间的（余弦）相似度会更高，这是向量检索的基础。
# 新版本路径是 langchain_openai（旧版 langchain.embeddings.openai 在新版下已不存在）。
from langchain_openai import OpenAIEmbeddings
# TODO: 请在此处补全代码（构造 OpenAIEmbeddings 实例）
embedding = None  # TODO

In [ ]:
# 三句话：前两句语义相近（狗/犬类是近义词），第三句语义完全不同（天气），
# 用来演示 embedding 向量能不能捕捉到这种语义相似度
sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

In [ ]:
# TODO: 请在此处补全代码
# 用 embedding.embed_query(...) 分别得到 sentence1/2/3 的向量
embedding1 = None  # TODO
embedding2 = None  # TODO
embedding3 = None  # TODO

In [ ]:
import numpy as np

In [ ]:
# 两个向量做点积（dot product）来衡量相似度：因为 OpenAI 的 embedding 向量是单位归一化过的，
# 点积等价于余弦相似度，值越接近 1 说明语义越相似
# 预期：dogs 和 canines 语义相近，点积应该比较高
np.dot(embedding1,embedding2)

In [ ]:
# dogs vs weather：语义无关，预期点积会明显低于上面 dogs vs canines 的结果
np.dot(embedding1,embedding3)

In [ ]:
# canines vs weather：同样语义无关，预期也偏低
np.dot(embedding2,embedding3)

## Vectorstores

In [ ]:
# 【真实 Bug 修复】Chroma：一个轻量级的本地向量数据库，用来存储 embedding 向量并支持相似度检索。
# 新版本路径是 langchain_community.vectorstores（旧版 langchain.vectorstores 已不存在）。
from langchain_community.vectorstores import Chroma

In [ ]:
# 向量库数据落盘的目录，重新运行 notebook 时可以直接从这里恢复，不用重新计算 embedding
persist_directory = 'docs/chroma/'

In [ ]:
!rm -rf ./docs/chroma  # remove old database files if any

In [ ]:
# TODO: 请在此处补全代码
# 用 Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory) 构造 vectordb
vectordb = None  # TODO

In [ ]:
# _collection.count() 是 Chroma 底层 collection 对象的方法，返回已经存进向量库的文档块数量
# 正常情况下应该等于 len(splits)
print(vectordb._collection.count())

### Similarity Search

In [ ]:
question = "is there an email i can ask for help"

In [ ]:
# TODO: 请在此处补全代码（用 vectordb.similarity_search(question, k=3) 得到 docs）
docs = None  # TODO

In [ ]:
# 应该返回 k=3 条结果
len(docs)

In [ ]:
docs[0].page_content

In [ ]:
# persist()：把内存里的向量库数据显式写盘到 persist_directory。
# 注意：在较新版本的 Chroma 里，只要构造时传了 persist_directory，数据其实已经是自动持久化的，
# 这里调用 persist() 更多是保留课程原有写法、确保写盘及时；即使不调用通常也不会报错。
vectordb.persist()

## Failure modes
朴素的相似度检索（similarity_search）不是万能的，下面演示两种常见的失败场景：
1. 数据里有重复文档时，检索结果会出现重复内容（缺乏多样性）；
2. 问题里带有明确的范围限定（比如"第三讲"），但纯语义检索无法保证只从对应来源里取内容（需要结合 metadata 过滤，见 04 节）。

In [ ]:
question = "what did they say about matlab?"

In [ ]:
# TODO: 请在此处补全代码（similarity_search，k=5，观察重复文档问题）
docs = None  # TODO

In [ ]:
docs[0]

In [ ]:
# 因为 Lecture01 被重复加载了两次，这里 docs[0] 和 docs[1] 很可能是同一段内容的两份重复结果，
# 这就是"失败模式一"：数据重复导致检索结果缺乏多样性（后面 04 节的 MMR 检索可以缓解这个问题）
docs[1]

In [ ]:
question = "what did they say about regression in the third lecture?"

In [ ]:
# TODO: 请在此处补全代码（similarity_search，k=5，观察"第三讲"限定条件失效问题）
docs = None  # TODO

In [ ]:
# 打印每条结果的来源（source 文件路径）：会发现结果并不都来自 "第三讲"（Lecture03），
# 说明纯向量相似度检索无法保证严格遵守问题里"第三讲"这个限定条件
for doc in docs:
    print(doc.metadata)

In [ ]:
# 看看第 5 条结果的具体内容，可能会发现它其实和"回归"这个主题关系不大，
# 这就是"失败模式二"：语义检索会返回"看起来相关但实际跑题"的内容
print(docs[4].page_content)